In [1]:
import pandas as pd

In [2]:
raw_reviews = pd.read_csv('raw_reviews.csv')
raw_reviews.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
raw_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


In [4]:
raw_reviews.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

In [5]:
raw_reviews.columns = raw_reviews.columns.str.lower()
raw_reviews.columns

Index(['unnamed: 0', 'clothing id', 'age', 'title', 'review text', 'rating',
       'recommended ind', 'positive feedback count', 'division name',
       'department name', 'class name'],
      dtype='object')

In [6]:
raw_reviews['rating'].value_counts(normalize=True).sort_values(ascending=False)

rating
5    0.559099
4    0.216171
3    0.122243
2    0.066635
1    0.035851
Name: proportion, dtype: float64

In [7]:
raw_reviews['rating'].mean()

np.float64(4.196031678446734)

In [8]:
# Calculate average review length
raw_reviews['review_length'] = raw_reviews['review text'].fillna('').apply(len)  
print(f"Average review length: {raw_reviews['review_length'].mean():.2f} characters")

Average review length: 297.58 characters


In [9]:
raw_reviews['review_length'].describe()

count    23486.000000
mean       297.581666
std        152.572686
min          0.000000
25%        173.000000
50%        292.000000
75%        451.000000
max        508.000000
Name: review_length, dtype: float64

In [10]:
#look at the longest review
raw_reviews.loc[raw_reviews['review_length']==508, 'review text'].values

array(['I adore this blouse. the colors are vibrant (see my photo below). this is one of my favorite purchases from retailer. the top is light weight. true to size. i ordered a petite small and am 5 feet tall 120 lbs. and curvy. i left it untucked and loose like in the photo and it was very flattering. i disagree about it being frumpy. i wore it with kelly green retailer brand slacks paired with the retailer yellow sweater coat with white piping, and the retailer moss suede cross bag and neutral color (nude) fl',
       "I have been continually disappointed in retailer's clothing for the past year. i feel like their prices have soared and the quality has remained so-so, but dresses like these somehow make the cut. it does not make sense. since when did retailer exclusively market to older women? no, retailer's market has always been young adults. this dress is extremely matronly. although the bottom print is nice, the hot pink belt?? is completely mismatched and gaudy. i really hope re

In [11]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Punkt is a multi-language, pre-trained, unsupervised sentence tokenizer in the NLTK library
nltk.download('punkt')

# List of stopwords in many spoken languages
nltk.download('stopwords')

# WordNet is a large, lexical database of the English language. Here it will handle the lemmatization
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/reizaimi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/reizaimi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/reizaimi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [12]:
sample = raw_reviews['review text'][600]

sample

"Great little transitional piece. i bought the blk/blk floral one & it's not your usual top. sort of quirky & one of a kind looking. i think it's crazy comfortable & i just love it! even its a bit chilly outdoors, toss on a scarf round one's neck & head on out. not the kind of top you'd wear with a jacket."

### Step 1 : Clean Text

In [13]:
def clean_text(text):
    if not isinstance(text, str):        # handle NaN values
        return ""
    text = text.lower()                  # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)         # remove punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()     # remove extra whitespace
    return text

# Test on sample review
sample = raw_reviews['review text'][600]
print("BEFORE:", sample)
print("AFTER: ", clean_text(sample))

BEFORE: Great little transitional piece. i bought the blk/blk floral one & it's not your usual top. sort of quirky & one of a kind looking. i think it's crazy comfortable & i just love it! even its a bit chilly outdoors, toss on a scarf round one's neck & head on out. not the kind of top you'd wear with a jacket.
AFTER:  great little transitional piece i bought the blkblk floral one its not your usual top sort of quirky one of a kind looking i think its crazy comfortable i just love it even its a bit chilly outdoors toss on a scarf round ones neck head on out not the kind of top youd wear with a jacket


### Step 2: Tokenize Text

In [14]:
def tokenize_text(text):
    return word_tokenize(text)

# Test on sample
cleaned_sample = clean_text(sample)
tokens = tokenize_text(cleaned_sample)

print("Cleaned text:", cleaned_sample)

Cleaned text: great little transitional piece i bought the blkblk floral one its not your usual top sort of quirky one of a kind looking i think its crazy comfortable i just love it even its a bit chilly outdoors toss on a scarf round ones neck head on out not the kind of top youd wear with a jacket


In [15]:
print("Tokens:", tokens)

Tokens: ['great', 'little', 'transitional', 'piece', 'i', 'bought', 'the', 'blkblk', 'floral', 'one', 'its', 'not', 'your', 'usual', 'top', 'sort', 'of', 'quirky', 'one', 'of', 'a', 'kind', 'looking', 'i', 'think', 'its', 'crazy', 'comfortable', 'i', 'just', 'love', 'it', 'even', 'its', 'a', 'bit', 'chilly', 'outdoors', 'toss', 'on', 'a', 'scarf', 'round', 'ones', 'neck', 'head', 'on', 'out', 'not', 'the', 'kind', 'of', 'top', 'youd', 'wear', 'with', 'a', 'jacket']


In [16]:
print("Number of tokens:", len(tokens))

Number of tokens: 58


### Step 3: Remove Stopwords

In [17]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

# Test on sample
filtered_tokens = remove_stopwords(tokens)

print("BEFORE stopword removal:", tokens)
print("AFTER stopword removal:", filtered_tokens)

BEFORE stopword removal: ['great', 'little', 'transitional', 'piece', 'i', 'bought', 'the', 'blkblk', 'floral', 'one', 'its', 'not', 'your', 'usual', 'top', 'sort', 'of', 'quirky', 'one', 'of', 'a', 'kind', 'looking', 'i', 'think', 'its', 'crazy', 'comfortable', 'i', 'just', 'love', 'it', 'even', 'its', 'a', 'bit', 'chilly', 'outdoors', 'toss', 'on', 'a', 'scarf', 'round', 'ones', 'neck', 'head', 'on', 'out', 'not', 'the', 'kind', 'of', 'top', 'youd', 'wear', 'with', 'a', 'jacket']
AFTER stopword removal: ['great', 'little', 'transitional', 'piece', 'bought', 'blkblk', 'floral', 'one', 'usual', 'top', 'sort', 'quirky', 'one', 'kind', 'looking', 'think', 'crazy', 'comfortable', 'love', 'even', 'bit', 'chilly', 'outdoors', 'toss', 'scarf', 'round', 'ones', 'neck', 'head', 'kind', 'top', 'youd', 'wear', 'jacket']


In [18]:
print(f"Tokens removed: {len(tokens) - len(filtered_tokens)}")

Tokens removed: 24


### Step 4: Lemmatize

In [19]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# Test on sample
lemmatized_tokens = lemmatize_tokens(filtered_tokens)

print("BEFORE lemmatization:", filtered_tokens)
print("AFTER lemmatization: ", lemmatized_tokens)

BEFORE lemmatization: ['great', 'little', 'transitional', 'piece', 'bought', 'blkblk', 'floral', 'one', 'usual', 'top', 'sort', 'quirky', 'one', 'kind', 'looking', 'think', 'crazy', 'comfortable', 'love', 'even', 'bit', 'chilly', 'outdoors', 'toss', 'scarf', 'round', 'ones', 'neck', 'head', 'kind', 'top', 'youd', 'wear', 'jacket']
AFTER lemmatization:  ['great', 'little', 'transitional', 'piece', 'bought', 'blkblk', 'floral', 'one', 'usual', 'top', 'sort', 'quirky', 'one', 'kind', 'looking', 'think', 'crazy', 'comfortable', 'love', 'even', 'bit', 'chilly', 'outdoors', 'toss', 'scarf', 'round', 'one', 'neck', 'head', 'kind', 'top', 'youd', 'wear', 'jacket']


In [30]:
def preprocess(text):
    text = clean_text(text)           # Step 1: clean
    tokens = tokenize_text(text)      # Step 2: tokenize
    tokens = remove_stopwords(tokens) # Step 3: remove stopwords
    tokens = lemmatize_tokens(tokens) # Step 4: lemmatize
    return tokens

# Apply to entire dataset
raw_reviews['tokens'] = raw_reviews['review text'].apply(preprocess)

In [31]:
# Preview results
raw_reviews[['review text', 'tokens']].head(5)

,review text,tokens
0,Absolutely wonderful - silky and sexy and comf...,"[absolutely, wonderful, silky, sexy, comfortable]"
1,Love this dress! it's sooo pretty. i happene...,"[love, dress, sooo, pretty, happened, find, st..."
2,I had such high hopes for this dress and reall...,"[high, hope, dress, really, wanted, work, init..."
3,"I love, love, love this jumpsuit. it's fun, fl...","[love, love, love, jumpsuit, fun, flirty, fabu..."
4,This shirt is very flattering to all due to th...,"[shirt, flattering, due, adjustable, front, ti..."


### Vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Join tokens back into strings
raw_reviews['clean_text'] = raw_reviews['tokens'].apply(lambda x: ' '.join(x))

,review text,clean_text
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love dress sooo pretty happened find store im ...
2,I had such high hopes for this dress and reall...,high hope dress really wanted work initially o...


In [32]:
# Preview
raw_reviews[['review text', 'clean_text']].head(3)

,review text,clean_text
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love dress sooo pretty happened find store im ...
2,I had such high hopes for this dress and reall...,high hope dress really wanted work initially o...


### Bag of Words (CountVectorizer)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(max_features=1000)

bow_matrix = bow_vectorizer.fit_transform(raw_reviews['clean_text'])

print("Bag of Words Matrix Shape:", bow_matrix.shape)
print("Sample feature names:", bow_vectorizer.get_feature_names_out()[:20])

Bag of Words Matrix Shape: (23486, 1000)
Sample feature names: ['able' 'absolutely' 'across' 'actually' 'add' 'added' 'addition'
 'adjustable' 'adorable' 'adore' 'afraid' 'ag' 'ago' 'agree' 'airy'
 'aline' 'almost' 'alone' 'along' 'already']


### TF-IDF (Term Frequency - Inverse Document Frequency)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=1000)

tfidf_matrix = tfidf_vectorizer.fit_transform(raw_reviews['clean_text'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print()
print("Sample feature names:", tfidf_vectorizer.get_feature_names_out()[:20])

TF-IDF Matrix Shape: (23486, 1000)

Sample feature names: ['able' 'absolutely' 'across' 'actually' 'add' 'added' 'addition'
 'adjustable' 'adorable' 'adore' 'afraid' 'ag' 'ago' 'agree' 'airy'
 'aline' 'almost' 'alone' 'along' 'already']


### Comparison: BoW vs TF-IDF

In [ ]:
# COMPARISON 

# Same word for reviews
word = 'dress'
bow_idx = list(bow_vectorizer.get_feature_names_out()).index(word)
tfidf_idx = list(tfidf_vectorizer.get_feature_names_out()).index(word)

print("Comparison for the word 'dress' in review 2 ===")
print(f"BoW value:   {bow_matrix[2, bow_idx]:.4f}  (raw count)")
print(f"TF-IDF value:{tfidf_matrix[2, tfidf_idx]:.4f}  (weighted score)")
print()
print("Key Differences")
print("BoW:   counts raw word frequency — simple but treats all words equally")
print("TFIDF: weights words by importance — common words across all reviews get penalized")

=== Comparison for the word 'dress' in review 2 ===
BoW value:   1.0000  (raw count)
TF-IDF value:0.0702  (weighted score)

=== Key Differences ===
BoW:   counts raw word frequency — simple but treats all words equally
TFIDF: weights words by importance — common words across all reviews get penalized


## Topic Extraction

In [29]:
#LDA
from sklearn.decomposition import LatentDirichletAllocation

# Define number of topics
n_topics = 5

# Fit LDA on BoW matrix
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(bow_matrix)

# Print top words per topic
print("LDA Topics")
feature_names = bow_vectorizer.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-10:]][::-1]
    print(f"Topic {i+1}: {', '.join(top_words)}")

LDA Topics
Topic 1: top, color, look, like, shirt, love, back, fabric, really, white
Topic 2: size, top, small, fit, like, im, would, ordered, look, large
Topic 3: size, fit, sweater, petite, color, length, sleeve, love, long, ordered
Topic 4: love, great, jean, wear, fit, perfect, pant, comfortable, color, bought
Topic 5: dress, fit, love, size, wear, flattering, beautiful, perfect, im, fabric


### NMF (Non-negative Matrix Factorization)

In [33]:
# NMF
from sklearn.decomposition import NMF

# Fit NMF on TF-IDF matrix
nmf = NMF(n_components=n_topics, random_state=42)
nmf.fit(tfidf_matrix)

# Print top words per topic
print("NMF Topics")
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
for i, topic in enumerate(nmf.components_):
    top_words = [feature_names_tfidf[j] for j in topic.argsort()[-10:]][::-1]
    print(f"Topic {i+1}: {', '.join(top_words)}")

NMF Topics
Topic 1: like, look, really, fabric, would, back, nice, shirt, much, color
Topic 2: dress, beautiful, flattering, love, perfect, wear, slip, comfortable, summer, fabric
Topic 3: love, great, jean, comfortable, color, fit, soft, perfect, wear, pant
Topic 4: size, small, fit, run, ordered, large, im, medium, usually, true
Topic 5: top, cute, bottom, love, pretty, beautiful, wear, tank, lace, white


In [34]:
# RESULTS 

print("""
=== Interpretation of Topics ===

LDA Topics:
- Topic 1: General appearance    → top, color, look, shirt, fabric
- Topic 2: Sizing issues         → size, small, fit, ordered, large
- Topic 3: Specific garments     → sweater, petite, sleeve, length
- Topic 4: Bottom wear           → jean, pant, comfortable, wear
- Topic 5: Dress reviews         → dress, flattering, beautiful, fabric

NMF Topics:
- Topic 1: General feedback      → like, look, fabric, shirt, color
- Topic 2: Dress reviews         → dress, beautiful, flattering, summer
- Topic 3: Bottom wear           → jean, comfortable, soft, pant
- Topic 4: Sizing & fit          → size, small, fit, ordered, medium
- Topic 5: Tops                  → top, cute, tank, lace, white

=== LDA vs NMF Comparison ===
- Both models identified similar themes: sizing, dresses, tops, bottoms
- NMF produced cleaner, more distinct topics with less word overlap
- LDA topics overlap more (e.g. 'size' and 'fit' appear in multiple topics)
- NMF is generally preferred for short texts like product reviews
""")


=== Interpretation of Topics ===

LDA Topics:
- Topic 1: General appearance    → top, color, look, shirt, fabric
- Topic 2: Sizing issues         → size, small, fit, ordered, large
- Topic 3: Specific garments     → sweater, petite, sleeve, length
- Topic 4: Bottom wear           → jean, pant, comfortable, wear
- Topic 5: Dress reviews         → dress, flattering, beautiful, fabric

NMF Topics:
- Topic 1: General feedback      → like, look, fabric, shirt, color
- Topic 2: Dress reviews         → dress, beautiful, flattering, summer
- Topic 3: Bottom wear           → jean, comfortable, soft, pant
- Topic 4: Sizing & fit          → size, small, fit, ordered, medium
- Topic 5: Tops                  → top, cute, tank, lace, white

=== LDA vs NMF Comparison ===
- Both models identified similar themes: sizing, dresses, tops, bottoms
- NMF produced cleaner, more distinct topics with less word overlap
- LDA topics overlap more (e.g. 'size' and 'fit' appear in multiple topics)
- NMF is general